# Retraining After Drift

Monitoring only matters if it leads to a responsible next step. In this notebook we close the loop: we evaluate the current production model on a fresh labeled batch, train a candidate on that new data, and promote the candidate only if it improves the holdout RMSE.

The notebook is designed to stay local-first. If you have not registered a production model yet, it will bootstrap one from the original January 2024 training month before running the comparison.


```mermaid
flowchart LR
    A["Current production<br>model"] --> B["Fresh labeled<br>batch"]
    B --> C["Holdout<br>evaluation"]
    C --> D["Train candidate on<br>current batch"]
    D --> E["Compare<br>RMSE"]
    E -->|better| F["Promote candidate"]
    E -->|worse| G["Keep current<br>production alias"]
```


## Imports and local configuration

We reuse the same feature set as the earlier notebooks so the retraining step stays comparable to the original baseline.


In [ ]:
import logging
import os
from pathlib import Path
from typing import Any
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline

# Show only errors from mlflow.utils; skip info and warning messages
logging.getLogger("mlflow.utils").setLevel(logging.ERROR)

In [ ]:
load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("green-taxi-monitoring")

color = "green"
model_name = "green-taxi-ride-duration"
baseline_year = 2024
baseline_month = 1
retrain_year = 2025
retrain_month = 5

# Require a small but real improvement before moving the production alias.
# This avoids promoting a candidate because of tiny random holdout variation.
min_rmse_improvement = 0.01

features = ["PULocationID", "DOLocationID", "trip_distance"]
target = "duration"

## Helper functions

These helpers keep the notebook compact and make the retraining workflow easier to read. The MLflow logging helper first logs the model, then registers it under the `green-taxi-ride-duration` name.


In [ ]:
def ensure_trip_data(year: int, month: int, color: str = "green") -> Path:
    """Download a public TLC batch once and reuse the local copy."""
    data_path = Path(f"./data/{color}_tripdata_{year}-{month:02d}.parquet")
    data_path.parent.mkdir(exist_ok=True)
    if not data_path.exists():
        urlretrieve(
            f"https://d37ci6vzurychx.cloudfront.net/trip-data/{color}_tripdata_{year}-{month:02d}.parquet",
            data_path,
        )
    return data_path


def preprocess(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Apply the same label, outlier, and category rules used by the baseline model."""
    df = raw_df.copy()
    df["duration"] = (
        df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]
    df = df[df["trip_distance"].between(0.1, 100)]
    df[["PULocationID", "DOLocationID"]] = df[["PULocationID", "DOLocationID"]].astype(
        str
    )
    return df[features + [target]]


def train_linear_model(X_train: pd.DataFrame, y_train: pd.Series) -> Pipeline:
    """One-hot encode the zones with DictVectorizer, then fit a linear regression."""
    model = make_pipeline(DictVectorizer(), LinearRegression())
    model.fit(X_train.to_dict(orient="records"), y_train)
    return model


def evaluate_model(
    model: Any, X_test: pd.DataFrame, y_test: pd.Series
) -> tuple[Any, float]:
    predictions = model.predict(X_test.to_dict(orient="records"))
    rmse = root_mean_squared_error(y_test, predictions)
    return predictions, float(rmse)


def create_registered_model_if_needed(client: MlflowClient, model_name: str) -> None:
    """Create the registry entry on a clean MLflow store and reuse it otherwise."""
    try:
        client.create_registered_model(
            name=model_name,
            tags={"framework": "scikit-learn"},
            description="Model to predict ride duration in NYC",
        )
    except MlflowException as exc:
        if exc.error_code != "RESOURCE_ALREADY_EXISTS":
            raise


def load_production_model(model_name: str) -> tuple[Any | None, str | None, str | None]:
    client = MlflowClient()
    try:
        model = mlflow.pyfunc.load_model(f"models:/{model_name}@production")
        model_version = client.get_model_version_by_alias(model_name, "production")
        return model, str(model_version.version), None
    except MlflowException as exc:
        # No registered model, no production alias or missing artifacts: let the caller bootstrap one.
        return None, None, str(exc)


def log_and_register_model(
    model: Pipeline,
    model_name: str,
    tags: dict[str, Any],
    metrics: dict[str, float],
    promote_to_production: bool,
) -> dict[str, Any]:
    with mlflow.start_run():
        mlflow.set_tags({key: str(value) for key, value in tags.items()})
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, float(metric_value))
        model_info = mlflow.sklearn.log_model(
            sk_model=model, name="mlflow-model", serialization_format="skops"
        )

        active_run = mlflow.active_run()
        assert active_run is not None
        run_id = active_run.info.run_id

        client = MlflowClient()
        create_registered_model_if_needed(client, model_name)
        model_version = client.create_model_version(
            name=model_name,
            source=model_info.model_uri,
            run_id=run_id,
        )

        promoted = False
        if promote_to_production:
            client.set_registered_model_alias(
                name=model_name,
                version=model_version.version,
                alias="production",
            )
            promoted = True

    return {
        "run_id": run_id,
        "model_version": int(model_version.version),
        "logged_model_uri": model_info.model_uri,
        "promoted": promoted,
    }

## Load or bootstrap the production model

If a `production` alias already exists in MLflow, we use it as the current baseline. If not, we create a baseline from the original January 2024 month so the rest of the notebook can still run end-to-end on a fresh local setup.


In [ ]:
production_model, production_version, load_error = load_production_model(model_name)
bootstrap_summary = None

if production_model is None:
    print(f"No production model loaded: {load_error}. Training a bootstrap baseline.")
    baseline_path = ensure_trip_data(baseline_year, baseline_month, color)
    baseline_df = preprocess(pd.read_parquet(baseline_path))
    baseline_X = baseline_df[features]
    baseline_y = baseline_df[target]
    baseline_X_train, baseline_X_test, baseline_y_train, baseline_y_test = (
        train_test_split(baseline_X, baseline_y, random_state=42, test_size=0.2)
    )
    bootstrap_model = train_linear_model(baseline_X_train, baseline_y_train)
    _, bootstrap_rmse = evaluate_model(
        bootstrap_model, baseline_X_test, baseline_y_test
    )
    bootstrap_result = log_and_register_model(
        model=bootstrap_model,
        model_name=model_name,
        tags={
            "developer": "<your-name>",
            "dataset": f"{color}-taxi",
            "year": baseline_year,
            "month": baseline_month,
            "purpose": "bootstrap-production-baseline",
        },
        metrics={"bootstrap_holdout_rmse": bootstrap_rmse},
        promote_to_production=True,
    )
    production_model, production_version, _ = load_production_model(model_name)
    bootstrap_summary = pd.DataFrame(
        [
            {
                "baseline_month": f"{baseline_year}-{baseline_month:02d}",
                "bootstrap_holdout_rmse": bootstrap_rmse,
                "production_version": bootstrap_result["model_version"],
            }
        ]
    )

assert production_model is not None
assert production_version is not None
bootstrap_summary

## Load the fresh labeled batch

We use `2025-05` here because it is a batch where drift may be worth investigating.


In [ ]:
current_path = ensure_trip_data(retrain_year, retrain_month, color)
df_current = pd.read_parquet(current_path)
df_processed = preprocess(df_current)
df_processed.head()

## Create a holdout split

The holdout rows stand in for delayed labels that arrive after the model has already been serving predictions. That gives us a fair comparison between the current production model and the retrained candidate.


In [ ]:
y = df_processed[target]
X = df_processed.drop(columns=[target])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

X_train.shape, X_test.shape

## Evaluate the current production model

This is the score the candidate has to beat before we consider a promotion.


In [ ]:
production_predictions, production_rmse = evaluate_model(
    production_model, X_test, y_test
)
production_rmse

## Train and evaluate the candidate model

We keep the model class deliberately simple. The point of this chapter is the retraining decision, not a feature engineering contest.


In [ ]:
candidate_model = train_linear_model(X_train, y_train)
candidate_predictions, candidate_rmse = evaluate_model(candidate_model, X_test, y_test)
candidate_rmse

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "production",
            "rmse": production_rmse,
            "production_version": production_version,
            "improvement_vs_production": 0.0,
        },
        {
            "model": "retrained_candidate",
            "rmse": candidate_rmse,
            "production_version": "new",
            "improvement_vs_production": production_rmse - candidate_rmse,
        },
    ]
)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparison["model"], comparison["rmse"], color=["#9aa0a6", "#1a73e8"])
ax.set_ylabel("RMSE")
ax.set_title("Production vs Retrained Candidate")
for i, value in enumerate(comparison["rmse"]):
    ax.text(i, value, f"{value:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

A lower RMSE means the model is closer to the true trip duration on unseen rides from the fresh batch. The most important question is not "do we retrain?" but "does retraining actually improve holdout performance?".


In [ ]:
sample_predictions = pd.DataFrame(
    {
        "actual": y_test.reset_index(drop=True).head(10),
        "production": pd.Series(production_predictions[:10]),
        "candidate": pd.Series(candidate_predictions[:10]),
    }
)
sample_predictions

## Register the candidate and decide on promotion

We still log every retraining attempt so the decision is traceable. This cell logs the model to MLflow, registers it under the shared model name, and moves the `production` alias only when the candidate improves RMSE by at least `min_rmse_improvement`.


In [ ]:
rmse_improvement = production_rmse - candidate_rmse
promote_candidate = rmse_improvement >= min_rmse_improvement

candidate_result = log_and_register_model(
    model=candidate_model,
    model_name=model_name,
    tags={
        "developer": "<your-name>",
        "dataset": f"{color}-taxi",
        "year": retrain_year,
        "month": retrain_month,
        "purpose": "retraining-after-drift",
        "promotion_rule": f"rmse_improvement >= {min_rmse_improvement}",
    },
    metrics={
        "production_rmse": production_rmse,
        "candidate_rmse": candidate_rmse,
        "rmse_improvement": rmse_improvement,
    },
    promote_to_production=promote_candidate,
)

In [ ]:
decision = pd.DataFrame(
    [
        {
            "production_version_before": production_version,
            "candidate_version": candidate_result["model_version"],
            "production_rmse": production_rmse,
            "candidate_rmse": candidate_rmse,
            "rmse_improvement": rmse_improvement,
            "min_rmse_improvement": min_rmse_improvement,
            "promoted": candidate_result["promoted"],
        }
    ]
)
decision

In [ ]:
if candidate_result["promoted"]:
    print(
        "The retrained candidate cleared the promotion threshold and is now the production alias."
    )
else:
    print(
        "The candidate was logged, but the production alias was left unchanged because the RMSE improvement was too small."
    )

print(f"Production version before comparison: {production_version}")
print(f"Candidate registered version: {candidate_result['model_version']}")
print(f"Production RMSE: {production_rmse:.3f}")
print(f"Candidate RMSE: {candidate_rmse:.3f}")
print(f"RMSE improvement: {rmse_improvement:.3f}")
print(f"Minimum improvement required: {min_rmse_improvement:.3f}")

## Takeaway

This notebook finishes the monitoring loop: detect drift, collect labeled feedback, retrain carefully, validate on holdout data, and only then promote. The key idea is that drift should trigger an evaluation workflow, not an automatic model swap.


## Cleanup

After you are done for the day:

- Stop all running terminals with `Ctrl + C`.

- Stop the services and delete the volumes with:

  ```bash
  docker compose down -v
  ```

To fully reset the repo to a clean local state, also run these commands from the repository root:

- Delete the MLflow runs, models, and data files:

  ```bash
  # macOS, Linux, Windows Git Bash
  rm -rf mlruns mlflow.db
  rm -f data/*.parquet
  ```

  ```powershell
  # Windows PowerShell
  Remove-Item -Recurse -Force mlruns, mlflow.db -ErrorAction SilentlyContinue
  Remove-Item data/*.parquet -ErrorAction SilentlyContinue
  ```

- Delete the local Prefect data (flow runs, deployments, logs):

  ```bash
  uv run prefect server database reset -y
  ```

> **Warning:** This database is shared by all your local Prefect projects, so their runs will get deleted too.
